# NV4 — Đánh giá hiệu suất mạng lưới trạm sạc

Notebook này dùng trực tiếp các hàm production trong `src/nv4_analysis.py`, `src/topsis.py` và `src/nv4_sensitivity.py`. Chạy cell theo thứ tự. Ledger ở bước 2 đối soát dữ liệu NV1 → NV4; cell cuối lưu bảng từ cùng lượt tính.

Quy tắc hiện hành: parser ngày và quy đổi PDT/PST/UTC theo NV1; ngưỡng công suất lấy từ `src.config.POWER_LIMITS_KW` theo `Port Type`; kiểm tra chênh lệch giữa thời lượng phiên và thời gian bắt đầu/kết thúc tối đa 59 giây. TOPSIS xếp theo điểm đầy đủ trước khi làm tròn; nếu điểm bằng nhau chính xác, tên trạm tăng dần là tiêu chí phụ.

## 1. Thiết lập

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
from scipy.stats import kendalltau, spearmanr
from IPython.display import display

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Open this notebook from the project workspace")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config import RAW_DATA_FILE, SESSIONS_CLEAN_PARQUET, POWER_LIMITS_KW
from src.nv4_analysis import NV4_CRITERIA, build_station_criteria, prepare_nv4_sessions
from src.nv4_sensitivity import run_sensitivity, summarize_rank_robustness
from src.topsis import entropy_weights, normalize_minmax, topsis

OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Source:", RAW_DATA_FILE)
print("Power limits by Port Type:", POWER_LIMITS_KW)

Source: /home/kedora/SEM5/ba/project/ev-charging-analytics---IS403/data/raw/EVChargingStationUsage.csv
Power limits by Port Type: {'Level 1': 1.92, 'Level 2': 22.0, 'default': 22.0}


## 2. Nạp dữ liệu và đối soát NV1 → NV4

Các dòng bị loại trong ledger là các bước tuần tự, không cộng trùng. Dòng `NV1 is_valid_session` là mốc đối chiếu độc lập. Dòng cuối cùng tách riêng số phiên bị NV4 loại thêm do kiểm tra timestamp.

In [2]:
raw = pd.read_csv(RAW_DATA_FILE, encoding="utf-8-sig", low_memory=False)
sessions, reconciliation, run_metadata = prepare_nv4_sessions(raw)
all_stations, X = build_station_criteria(sessions)
run_metadata.update({
    "station_count_after_cleaning": len(all_stations),
    "eligible_station_count": len(X),
    "sessions_in_eligible_stations": int(X["C1_so_phien"].sum()),
})
if SESSIONS_CLEAN_PARQUET.exists():
    nv1_clean = pd.read_parquet(SESSIONS_CLEAN_PARQUET)
    parquet_valid = int(nv1_clean["is_valid_session"].sum())
    if parquet_valid != run_metadata["nv1_valid_sessions"]:
        raise AssertionError(f"NV1 count mismatch: parquet={parquet_valid}, raw={run_metadata['nv1_valid_sessions']}")
    run_metadata["sessions_clean_parquet_valid_sessions"] = parquet_valid

reconciliation = pd.concat([reconciliation, pd.DataFrame([
    {"stage": "Distinct stations after NV4 filters", "rows_remaining": len(all_stations),
     "removed_at_stage": None, "rule": "Unique normalized Station Name"},
    {"stage": "Stations eligible for ranking", "rows_remaining": len(X),
     "removed_at_stage": len(all_stations)-len(X), "rule": "At least 500 NV4-valid sessions"},
    {"stage": "Sessions in eligible stations", "rows_remaining": int(X.C1_so_phien.sum()),
     "removed_at_stage": len(sessions)-int(X.C1_so_phien.sum()),
     "rule": "Sessions in stations meeting the threshold"},
])], ignore_index=True)
display(reconciliation)
display(pd.Series(run_metadata, name="value").to_frame())

,stage,rows_remaining,removed_at_stage,rule
0,Raw CSV rows,259415,NaN,Raw source rows
1,After exact duplicate removal,259411,4.0,Exact duplicate rows removed
2,After positive/consistent duration checks,259366,45.0,"Require total > 0, charging > 0, charging <= t..."
3,After Port Type power limits,259356,10.0,Use POWER_LIMITS_KW from src.config by Port Type
4,After timestamp consistency,258875,481.0,NV1 date parser + UTC; absolute duration diffe...
5,NV1 is_valid_session (comparison),259356,NaN,charging <= total and power <= configured Port...
6,NV1-valid sessions excluded by NV4 timestamp c...,481,NaN,Only the additional NV4 timestamp consistency ...
7,Distinct stations after NV4 filters,46,NaN,Unique normalized Station Name
8,Stations eligible for ranking,33,13.0,At least 500 NV4-valid sessions
9,Sessions in eligible stations,258836,39.0,Sessions in stations meeting the threshold


,value
raw_rows,259415
exact_duplicates_removed,4
nv1_valid_sessions,259356
nv4_valid_sessions,258875
nv4_timestamp_rejected_after_other_checks,481
power_limits_kw,"{'Level 1': 1.92, 'Level 2': 22.0, 'default': ..."
timestamp_tolerance_seconds,59
station_count_after_cleaning,46
eligible_station_count,33
sessions_in_eligible_stations,258836


## 3. Tiêu chí và tương quan

In [3]:
criterion_columns = list(NV4_CRITERIA)
directions = {c: NV4_CRITERIA[c][2] for c in criterion_columns}
labels = {c: NV4_CRITERIA[c][0] for c in criterion_columns}
decision = X[criterion_columns]
corr_pearson = decision.corr(method="pearson")
corr_spearman = decision.corr(method="spearman")
display(decision.head())
display(corr_pearson.round(4))
display(corr_spearman.round(4))
print("Spearman(C5,C6):", corr_spearman.loc["C5_ty_le_nhan_roi", "C6_ty_le_chiem_dung_nghiem_trong"])

,C1_so_phien,C2_tong_dien_nang,C3_dien_nang_moi_phien,C4_gio_sac_moi_phien,C5_ty_le_nhan_roi,C6_ty_le_chiem_dung_nghiem_trong
Station Name,,,,,,
PALO ALTO CA / HAMILTON #2,23682,191268.391057,8.076530,2.032521,0.120082,0.077358
PALO ALTO CA / HAMILTON #1,14836,113384.729360,7.642540,2.042863,0.126852,0.082569
PALO ALTO CA / WEBSTER #1,14408,138831.830993,9.635746,2.146363,0.129359,0.086063
PALO ALTO CA / BRYANT #2,14296,106687.255028,7.462735,2.0014,0.123028,0.086877
PALO ALTO CA / HIGH #4,13844,111443.448651,8.049946,1.947649,0.119726,0.078662


,C1_so_phien,C2_tong_dien_nang,C3_dien_nang_moi_phien,C4_gio_sac_moi_phien,C5_ty_le_nhan_roi,C6_ty_le_chiem_dung_nghiem_trong
C1_so_phien,1.0000,0.9815,-0.2814,0.0610,-0.0531,-0.1425
C2_tong_dien_nang,0.9815,1.0000,-0.1254,0.1462,-0.0532,-0.1466
C3_dien_nang_moi_phien,-0.2814,-0.1254,1.0000,0.7377,-0.0914,-0.0874
C4_gio_sac_moi_phien,0.0610,0.1462,0.7377,1.0000,0.1286,0.0758
C5_ty_le_nhan_roi,-0.0531,-0.0532,-0.0914,0.1286,1.0000,0.9853
C6_ty_le_chiem_dung_nghiem_trong,-0.1425,-0.1466,-0.0874,0.0758,0.9853,1.0000


,C1_so_phien,C2_tong_dien_nang,C3_dien_nang_moi_phien,C4_gio_sac_moi_phien,C5_ty_le_nhan_roi,C6_ty_le_chiem_dung_nghiem_trong
C1_so_phien,1.0000,0.9863,-0.2045,-0.0231,0.3840,0.3406
C2_tong_dien_nang,0.9863,1.0000,-0.0969,0.0354,0.3981,0.3653
C3_dien_nang_moi_phien,-0.2045,-0.0969,1.0000,0.6922,-0.1437,-0.1170
C4_gio_sac_moi_phien,-0.0231,0.0354,0.6922,1.0000,0.1965,0.1487
C5_ty_le_nhan_roi,0.3840,0.3981,-0.1437,0.1965,1.0000,0.9709
C6_ty_le_chiem_dung_nghiem_trong,0.3406,0.3653,-0.1170,0.1487,0.9709,1.0000


Spearman(C5,C6): 0.9709224598930482


## 4. Min-Max, Entropy và TOPSIS

In [4]:
R = normalize_minmax(decision, directions)
entropy_table, w_entropy = entropy_weights(R, labels)
w_equal = pd.Series(1.0 / len(criterion_columns), index=criterion_columns, name="Trọng số đều")
display(entropy_table.round(6))

top_entropy = topsis(R, w_entropy)
top_equal = topsis(R, w_equal)
df_rank = pd.DataFrame({
    "Station Name": X.index.to_numpy(),
    "C1_so_phien": X["C1_so_phien"].to_numpy(),
    "Score_Entropy": top_entropy["TOPSIS_Score"].to_numpy(),
    "Rank_Entropy": top_entropy["Rank"].to_numpy(),
    "Score_Equal": top_equal["TOPSIS_Score"].to_numpy(),
    "Rank_Equal": top_equal["Rank"].to_numpy(),
})
df_rank["Rank_Diff"] = (df_rank.Rank_Entropy - df_rank.Rank_Equal).abs()
df_rank = df_rank.sort_values(["Rank_Entropy", "Station Name"], kind="mergesort").reset_index(drop=True)
display(df_rank)
display(df_rank.head(5))
display(df_rank.tail(5))
print("Baseline Spearman:", spearmanr(df_rank.Rank_Entropy, df_rank.Rank_Equal).statistic)
print("Baseline Kendall:", kendalltau(df_rank.Rank_Entropy, df_rank.Rank_Equal).statistic)
print("Mean/max rank difference:", df_rank.Rank_Diff.mean(), df_rank.Rank_Diff.max())

,Entropy (e_j),Độ phân kỳ (d_j = 1 - e_j),Trọng số Entropy (w_j),Tên tiêu chí
Tiêu chí,,,,
C1_so_phien,0.920431,0.079569,0.294388,Số phiên sạc
C2_tong_dien_nang,0.924948,0.075052,0.277676,Tổng điện năng
C3_dien_nang_moi_phien,0.943857,0.056143,0.207718,Điện năng bình quân mỗi phiên
C4_gio_sac_moi_phien,0.973064,0.026936,0.099659,Thời gian sạc hữu ích bình quân
C5_ty_le_nhan_roi,0.982417,0.017583,0.065053,Tỷ lệ chiếm dụng nhàn rỗi
C6_ty_le_chiem_dung_nghiem_trong,0.984997,0.015003,0.055507,Tỷ lệ phiên chiếm dụng nghiêm trọng


,Station Name,C1_so_phien,Score_Entropy,Rank_Entropy,Score_Equal,Rank_Equal,Rank_Diff
0,PALO ALTO CA / MPL #6,8979,0.723298,1,0.672263,2,1
1,PALO ALTO CA / MPL #2,1531,0.647613,2,0.674869,1,1
2,PALO ALTO CA / MPL #3,1420,0.604583,3,0.647787,3,0
3,PALO ALTO CA / CAMBRIDGE #2,9245,0.512089,4,0.555134,9,5
4,PALO ALTO CA / HIGH #2,6921,0.506990,5,0.554710,10,5
5,PALO ALTO CA / MPL #1,1044,0.492506,6,0.584513,6,0
6,PALO ALTO CA / HAMILTON #1,14836,0.483649,7,0.533183,16,9
7,PALO ALTO CA / HIGH #3,8311,0.478468,8,0.559798,8,0
8,PALO ALTO CA / CAMBRIDGE #4,3426,0.472860,9,0.610314,4,5
9,PALO ALTO CA / HAMILTON #2,23682,0.471556,10,0.543703,13,3


,Station Name,C1_so_phien,Score_Entropy,Rank_Entropy,Score_Equal,Rank_Equal,Rank_Diff
0,PALO ALTO CA / MPL #6,8979,0.723298,1,0.672263,2,1
1,PALO ALTO CA / MPL #2,1531,0.647613,2,0.674869,1,1
2,PALO ALTO CA / MPL #3,1420,0.604583,3,0.647787,3,0
3,PALO ALTO CA / CAMBRIDGE #2,9245,0.512089,4,0.555134,9,5
4,PALO ALTO CA / HIGH #2,6921,0.506990,5,0.554710,10,5


,Station Name,C1_so_phien,Score_Entropy,Rank_Entropy,Score_Equal,Rank_Equal,Rank_Diff
28,PALO ALTO CA / TED THOMPSON #1,9804,0.196495,29,0.419360,28,1
29,PALO ALTO CA / HIGH #1,10746,0.189231,30,0.419249,29,1
30,PALO ALTO CA / RINCONADA LIB 3,6492,0.187964,31,0.280276,32,1
31,PALO ALTO CA / BRYANT #6,5848,0.179308,32,0.281993,31,1
32,PALO ALTO CA / WEBSTER #3,10467,0.170144,33,0.404260,30,3


Baseline Spearman: 0.8191844919786097
Baseline Kendall: 0.6363636363636364
Mean/max rank difference: 4.303030303030303 14


## 5. Kiểm tra độ nhạy S0–S6

In [5]:
df_sensitivity = run_sensitivity(X, NV4_CRITERIA)
display(df_sensitivity)
rank_robustness = summarize_rank_robustness(X, NV4_CRITERIA, df_sensitivity)
display(pd.Series({k: v for k, v in rank_robustness.items() if k != "scenarios"}, name="value").to_frame())
display(pd.DataFrame(rank_robustness["scenarios"]).T)


,Scenario,Criteria_Count,Spearman_Entropy,Kendall_Entropy,Mean_Rank_Diff_Entropy,Max_Rank_Diff_Entropy,Stations_Diff_ge5_Entropy,Top5_Overlap_Entropy,Spearman_Equal,Kendall_Equal,Mean_Rank_Diff_Equal,Max_Rank_Diff_Equal,Stations_Diff_ge5_Equal,Top5_Overlap_Equal
0,S0: Gốc (C1-C6),6,1.0000,1.0000,0.00,0,0,5/5,1.0000,1.0000,0.00,0.0,0.0,5/5
1,S1: Loại C2 (Tổng kWh),5,0.9559,0.8447,2.00,8,3,4/5,0.9178,0.8030,2.85,9.0,8.0,4/5
2,S2: Loại C1 (Số phiên),5,0.9261,0.7727,2.85,11,7,3/5,0.8927,0.7803,3.15,12.0,9.0,4/5
3,S3: Loại C6 (Chiếm dụng nặng),5,0.9970,0.9811,0.30,2,0,5/5,0.9890,0.9432,0.73,5.0,1.0,5/5
4,S4: Loại C4 (Thời gian sạc),5,0.9920,0.9432,0.85,3,0,5/5,0.9208,0.7841,2.55,10.0,6.0,4/5
5,S5: C5 trọng số thời lượng,6,0.9987,0.9886,0.18,2,0,5/5,0.9923,0.9508,0.73,4.0,0.0,5/5
6,S6: Vector Normalization,6,0.8870,0.7311,3.09,16,6,5/5,NaN,NaN,NaN,NaN,NaN,N/A


,value
station_count,33
correlation_c1_c2,0.981503
correlation_c5_c6,0.985274
entropy_weight_c1_c2_sum,0.572064
interpretation,"Ranking is conditional on criteria, weights an..."


,spearman_entropy,stations_shifted_at_least_5_ranks,max_rank_shift,top5_overlap
S1,0.9559,3,8,4/5
S2,0.9261,7,11,3/5
S6,0.887,6,16,5/5


## 6. Lưu bảng kết quả

In [6]:
reconciliation.to_csv(OUTPUT_DIR / "nv4_session_reconciliation.csv", index=False, encoding="utf-8-sig")
pd.Series(run_metadata, name="value").to_json(OUTPUT_DIR / "nv4_run_metadata.json", indent=2, force_ascii=False)
X[criterion_columns].to_csv(OUTPUT_DIR / "nv4_station_criteria.csv", encoding="utf-8-sig")
entropy_table.to_csv(OUTPUT_DIR / "nv4_criteria_weights.csv", encoding="utf-8-sig")
df_rank.to_csv(OUTPUT_DIR / "nv4_topsis_rankings.csv", index=False, encoding="utf-8-sig")
corr_pearson.to_csv(OUTPUT_DIR / "nv4_criteria_correlations_pearson.csv", encoding="utf-8-sig")
corr_spearman.to_csv(OUTPUT_DIR / "nv4_criteria_correlations_spearman.csv", encoding="utf-8-sig")
df_sensitivity.to_csv(OUTPUT_DIR / "nv4_sensitivity_analysis.csv", index=False, encoding="utf-8-sig")
print("Saved NV4 tables from this run to", OUTPUT_DIR)
(OUTPUT_DIR / "nv4_rank_robustness.json").write_text(json.dumps(rank_robustness, ensure_ascii=False, indent=2), encoding="utf-8")


Saved NV4 tables from this run to /home/kedora/SEM5/ba/project/ev-charging-analytics---IS403/outputs


778